# Notebook 9 — Vedere lo shift e adattarsi (digits)

Il Notebook 4 si fermava a *misurare* lo shift (AUROC, rejection curve,
rotazione di MNIST) senza mai adattare il modello. Qui si fa un passo in
più: dopo aver confermato lo shift su MNIST/USPS (stessa idea del Notebook
4, con `src/metrics.py`), si adatta davvero il modello con
`adapt_target`/`im_loss` (`src/digits_adapt.py`, portati da
`src/im_adapt.py` del progetto principale -- `code_v2` non aveva un
adattamento vero e proprio, solo la misura dello shift).

Convenzione: `g` sbloccato, `h` congelato (gestito da `adapt_target`
stesso), `weight_mode="uncertainty"` (default della funzione, U-SFAN),
`gamma=0.5`, `temperature=0.4`, `lr=1e-2`, `M=100`, `steps=300` -- stessa
convenzione già usata per HAR nel progetto principale. **Prima di lanciare
i 300 step per intero**, misura il tempo di un singolo step full-batch
sulla dimensione reale del dominio target (`steps=1` più sotto): se il
costo non è ragionevole su questa architettura (più piccola di quella usata
in una versione precedente di questo stesso esperimento), valuta un
sottocampionamento invece di assumere che vada bene.

## Setup: ricarica checkpoint, fit di Laplace

In [ ]:
import sys, copy, time
from pathlib import Path

cwd = Path().resolve()
PROJ = cwd
while not (PROJ / "src").exists():
    PROJ = PROJ.parent
sys.path.insert(0, str(PROJ))

import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader, TensorDataset

from src.digits_data import load_domain
from src.digits_model import SmallCNN32
from src.bayesian_model import extract, augment, LastLayerLaplace
from src.digits_adapt import adapt_target
from src.metrics import auroc, rejection_curve

MODELS_DIR = PROJ / "models" / "source_svhn"
ckpt = torch.load(MODELS_DIR / "model.pt", map_location="cpu", weights_only=False)
lap_data = np.load(MODELS_DIR / "svhn_laplace.npz")
mc_conv = np.load(MODELS_DIR / "mc_convergence.npz")
mean, std = ckpt["source_mean"], ckpt["source_std"]
M_FIXED = int(mc_conv["M_FIXED"])

model = SmallCNN32(n_classes=ckpt["n_classes"], feature_dim=ckpt["feature_dim"])
model.load_state_dict(ckpt["state_dict"])
model.eval()
laplace = LastLayerLaplace(theta_map=lap_data["theta_map"], cov=lap_data["cov"],
                           K=int(lap_data["K"]), Dp=int(lap_data["Dp"]))

TARGET_DOMAINS = ["mnist", "usps"]
target_raw = {d: load_domain(d, "test", mean, std) for d in TARGET_DOMAINS}
print(f"M_FIXED (dal Notebook 7) = {M_FIXED}")

## 1. Vedere lo shift (prima dell'adattamento)

Stessa idea del Notebook 4: usa l'incertezza epistemica come score per
rilevare i punti fuori dal manifold source (AUROC, con "positivo" = target)
e la rejection curve (accuratezza in funzione della copertura, scartando i
punti più incerti).

In [ ]:
def predict_domain(X, y):
    loader = DataLoader(TensorDataset(X, y), batch_size=256)
    Phi, y_np, _ = extract(model, loader, device="cpu")
    Phi_aug = augment(Phi)
    rng = np.random.default_rng(456)
    pred = laplace.predictive(Phi_aug, M=M_FIXED, rng=rng)
    return pred, y_np


X_svhn_test, y_svhn_test = load_domain("svhn", "test", mean, std)
pred_svhn, y_svhn_np = predict_domain(X_svhn_test, y_svhn_test)

pre_results = {}
for domain in TARGET_DOMAINS:
    X_t, y_t = target_raw[domain]
    pred_t, y_t_np = predict_domain(X_t, y_t)
    pre_results[domain] = dict(pred=pred_t, y=y_t_np)

    scores = np.concatenate([pred_svhn["epistemic"], pred_t["epistemic"]])
    labels = np.concatenate([np.zeros(len(y_svhn_np)), np.ones(len(y_t_np))])
    a = auroc(scores, labels)
    acc_pre = (pred_t["probs"].argmax(axis=1) == y_t_np).mean()
    print(f"{domain}: AUROC(epistemica, svhn vs {domain})={a:.3f}  accuracy pre-adattamento={100*acc_pre:.2f}%")

## 2. Misura del costo di un passo full-batch, poi decisione

In [ ]:
def benchmark_one_step(X, y):
    m = copy.deepcopy(model)
    t0 = time.time()
    adapt_target(m, laplace, X, weight_mode="uncertainty", gamma=0.5, temperature=0.4,
                lr=1e-2, steps=1, M=100, seed=0)
    return time.time() - t0


for domain in TARGET_DOMAINS:
    X_t, y_t = target_raw[domain]
    elapsed = benchmark_one_step(X_t, y_t)
    print(f"{domain}: 1 step full-batch ({X_t.shape[0]} immagini) = {elapsed:.2f}s")

**Decisione** (da confermare guardando i tempi sopra prima di eseguire la
cella seguente per intero): se pochi secondi/step, procedi full-batch con
`steps=300` come sotto, fedele alla convenzione HAR. Se invece il costo
risultasse alto (dell'ordine dei minuti/step, come capitato con un backbone
molto più grande in una versione precedente di questo esperimento), riduci
`ADAPT_STEPS` o introduci un sottocampionamento fisso, documentandolo qui
prima di procedere.

## 3. Adattamento: `adapt_target` su MNIST test e USPS test

In [ ]:
ADAPT_STEPS = 300  # convenzione HAR -- rivedi in base alla cella di benchmark sopra
ADAPT_KWARGS = dict(weight_mode="uncertainty", gamma=0.5, temperature=0.4, lr=1e-2, M=100)

adaptation_results = {}
for domain, seed in [("mnist", 1), ("usps", 2)]:
    print(f"\n{'=' * 60}\nAdattamento su {domain} (full batch, steps={ADAPT_STEPS}, seed={seed})\n{'=' * 60}")
    X_t, y_t = target_raw[domain]
    m = copy.deepcopy(model)
    hist = adapt_target(m, laplace, X_t, steps=ADAPT_STEPS, seed=seed, **ADAPT_KWARGS)

    m.eval()
    with torch.no_grad():
        probs_post = torch.softmax(m(X_t), dim=-1).numpy()
    with torch.no_grad():
        phi_post = m.features(X_t).numpy()
    Phi_aug_post = augment(phi_post)

    acc_pre = (pre_results[domain]["pred"]["probs"].argmax(axis=1) == pre_results[domain]["y"]).mean()
    acc_post = (probs_post.argmax(axis=1) == y_t.numpy()).mean()

    adaptation_results[domain] = dict(hist=hist, probs_post=probs_post, Phi_aug_post=Phi_aug_post,
                                      y=y_t.numpy(), acc_pre=acc_pre, acc_post=acc_post)
    print(f"\n{domain}: pre={100*acc_pre:.2f}%  post={100*acc_post:.2f}%  "
          f"delta={100*(acc_post-acc_pre):+.2f}pp")

## Traiettoria di loss/entropia/diversità

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
fields = ["loss", "ent", "div", "mean_weight"]
titles = ["loss IM", "termine entropia (pesato)", "termine diversità", "peso medio per campione"]
colors = {"mnist": "tab:purple", "usps": "tab:red"}
for ax, field, title in zip(axes, fields, titles):
    for domain in TARGET_DOMAINS:
        ax.plot(adaptation_results[domain]["hist"][field], color=colors[domain], label=domain)
    ax.set_xlabel("step di adattamento")
    ax.set_title(title, fontsize=10)
    ax.grid(True, alpha=0.3)
axes[0].legend(fontsize=8)
fig.suptitle(f"Traiettorie di adattamento (steps={ADAPT_STEPS}, full batch)")
fig.tight_layout()
plt.show()

## 4. BALD e calibrazione dopo l'adattamento

In [ ]:
from src.metrics import reliability_bins, expected_calibration_error

post_results = {}
for domain in TARGET_DOMAINS:
    r = adaptation_results[domain]
    rng = np.random.default_rng(456)
    pred_post = laplace.predictive(r["Phi_aug_post"], M=M_FIXED, rng=rng)
    ece_post = expected_calibration_error(r["probs_post"], r["y"])
    post_results[domain] = dict(pred=pred_post, ece=ece_post)

    epi_pre = pre_results[domain]["pred"]["epistemic"].mean()
    epi_post = pred_post["epistemic"].mean()
    ece_pre = expected_calibration_error(pre_results[domain]["pred"]["probs"], pre_results[domain]["y"])
    print(f"{domain}: epistemica pre={epi_pre:.4f} post={epi_post:.4f} delta={epi_post-epi_pre:+.4f} | "
          f"ECE pre={ece_pre:.4f} post={ece_post:.4f} delta={ece_post-ece_pre:+.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
for ax, domain in zip(axes, TARGET_DOMAINS):
    data = [pre_results[domain]["pred"]["epistemic"], post_results[domain]["pred"]["epistemic"]]
    parts = ax.violinplot(data, showmeans=True, showextrema=True)
    for pc, color in zip(parts["bodies"], ["tab:gray", "tab:orange"]):
        pc.set_facecolor(color); pc.set_alpha(0.6)
    ax.set_xticks([1, 2]); ax.set_xticklabels(["pre-adattamento", "post-adattamento"])
    ax.set_ylabel("epistemica (nats)")
    ax.set_title(domain)
    ax.grid(True, alpha=0.3, axis="y")
fig.suptitle(f"Epistemica pre vs. post adattamento (steps={ADAPT_STEPS}, full batch)")
fig.tight_layout()
plt.show()

## Riepilogo pre/post adattamento

In [ ]:
print(f"{'dominio':>10s} {'pre':>10s} {'post':>10s} {'delta':>10s}")
print("-" * 44)
for domain in TARGET_DOMAINS:
    r = adaptation_results[domain]
    print(f"{domain:>10s} {100*r['acc_pre']:9.2f}% {100*r['acc_post']:9.2f}% "
          f"{100*(r['acc_post']-r['acc_pre']):+9.2f}pp")

## TODO prima che questo sia un risultato di tesi

Questo intero notebook usa **un solo seed di training del source (2019) e
una sola run per condizione** -- un solo fit di Laplace, una sola run di
adattamento per dominio target, nessuna ripetizione. Ogni numero e ogni
delta pre/post sopra (accuracy, epistemica, ECE) è quindi una singola
stima puntuale, non una distribuzione: da questo notebook da solo non si
può distinguere un effetto reale dal rumore di ottimizzazione/Monte Carlo
di quella singola run.

**Prima di riportare uno qualunque di questi confronti pre/post, o
qualunque correlazione costruita sopra, come risultato definitivo**, va
rifatto con **almeno 5 seed** -- variando sia il seed di training del
source sia il seed di adattamento -- seguendo la stessa convenzione già
usata per HAR (`SEEDS = [0, 1, 2, 3, 4]`), riportando media ± deviazione
standard sulle run.